# Problem Set 5: Autoencoders
## CHEME 5820 · Machine Learning and Artificial Intelligence for Engineers · Spring 2025

**Name**: _______________________   **NetID**: _______________   **Date**: _______________

---

In this problem set you will implement and train a **deterministic Autoencoder (AE)** on
MNIST handwritten digit images. An autoencoder is a neural network that learns to
compress data into a low-dimensional **bottleneck code** and then reconstruct the original
from that code alone — a direct neural-network analogue of the embedding models (CBOW,
Skip-Gram) you built in lecture.

> __Learning Objectives__
>
> By the end of this problem set, you should be able to:
> * __Implement an encoder and decoder using Flux.jl:__ Write `encode` and `decode` functions that map inputs to a low-dimensional bottleneck and back, using `Chain` and `Dense` layers.
> * __Implement and minimise a reconstruction loss:__ Compute mean-squared error between the input and the reconstruction, and write a Flux.jl training loop that minimises it.
> * __Visualise and interpret the learned latent space:__ Project bottleneck codes onto two principal components and interpolate between pairs of images in latent space.

| Problem | Topic | Points |
|---------|-------|--------|
| 1 | Data loading & exploration | 15 |
| 2 | AE architecture | 30 |
| 3 | Reconstruction loss & training | 30 |
| 4 | Latent space analysis | 25 |
| **Total** | | **100** |

## Setup, Data, and Prerequisites
We set up the computational environment by including the `Include.jl` file, loading the
MNIST dataset, and setting up the required constants.

> __Environment Setup with Include.jl__
>
> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of `Include.jl` in the notebook's global scope. `Include.jl` sets paths, loads required external packages, and includes `src/Types.jl` and `src/Compute.jl`, which define the `MyAEModel` type and the helper functions used throughout this notebook.

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

In addition to standard Julia libraries, we use [Flux.jl](https://fluxml.ai/Flux.jl/stable/) for automatic differentiation and neural network layers, [MLDatasets.jl](https://juliaml.github.io/MLDatasets.jl/stable/) to load MNIST, and [Plots.jl](https://docs.juliaplots.org/stable/) for visualization.

### Implementations
The notebook uses the following helper functions from `src/Compute.jl`. Student-implemented functions (`encode`, `decode`, `reconstruction_loss`) are defined directly in notebook cells below.

| Function | Source | Description |
|----------|--------|-------------|
| `build_ae_model(input_dim, hidden_dim, latent_dim)` | `src/Compute.jl` | Constructs a `MyAEModel` with a symmetric encoder–decoder architecture. |
| `load_mnist_digit(digit; n_examples)` | `src/Compute.jl` | Loads MNIST training images for one digit class; returns a `(784 × N)` Float32 matrix. |
| `show_image_grid(X; nrows, ncols)` | `src/Compute.jl` | Displays columns of a `784 × N` matrix as a grid of 28×28 greyscale images. |

### Constants
Let's set some constants that we will use throughout. See comments for permissible values.

In [ ]:
DIGIT         = 3;       # MNIST digit class to model (0–9)
K             = 100;     # number of training examples
D             = 784;     # input dimension: 28 × 28 = 784 pixels
L             = 8;       # bottleneck (latent) dimension
H             = 256;     # hidden-layer width
LR            = 1f-3;    # Adam learning rate
NUM_EPOCHS    = 2_000;   # training epochs

Random.seed!(42);

___
## Background: Autoencoders

An autoencoder is a neural network trained to reproduce its own input at the output layer,
subject to passing through a low-dimensional **bottleneck**. It consists of two sub-networks:

| Component | Maps | Role |
|-----------|------|------|
| **Encoder** $f_\theta$ | $\mathbf{x}\in\mathbb{R}^D \to \mathbf{z}\in\mathbb{R}^L,\; L\ll D$ | compresses input to a compact code |
| **Decoder** $g_\phi$ | $\mathbf{z}\in\mathbb{R}^L \to \hat{\mathbf{x}}\in\mathbb{R}^D$ | reconstructs the input from the code |

Both are trained jointly to minimise the **reconstruction loss** over the training set:

$$\mathcal{L}(\theta,\phi) = \frac{1}{N}\sum_{i=1}^{N}\|\mathbf{x}_i - g_\phi(f_\theta(\mathbf{x}_i))\|^2$$

Because the only path from input to output passes through the $L$-dimensional bottleneck,
the encoder is forced to retain only the most important structure of the data — and the
decoder must learn to recover the full input from that compressed representation.

> __Connection to CBOW and Skip-Gram__
>
> You have already built models with this compress-then-reconstruct structure. In CBOW,
> the input weight matrix $\mathbf{W}_1$ acts as an encoder that maps a sparse one-hot
> word vector into a dense $d_h$-dimensional embedding, and $\mathbf{W}_2$ decodes that
> embedding back into a prediction over the vocabulary. An autoencoder is the same idea
> applied to continuous inputs (images) with deeper, non-linear encoder and decoder networks.

___
## Problem 1 — Data Loading and Exploration  (15 points)

We train the autoencoder on $K = 100$ examples of MNIST digit **3** from the training
split. Each 28×28 greyscale image is flattened to a 784-dimensional vector and stored as a
column of the data matrix $\mathbf{X}\in\mathbb{R}^{784\times K}$, with pixel values in
$[0, 1]$.

### 1a  (5 pts)
Call `load_mnist_digit` to load the data, store the result in `X`, and print its size.
**Expected: `(784, 100)`**.

### 1b  (5 pts)
Compute and print the mean and standard deviation of all pixel values in `X`.

### 1c  (5 pts)
Call `show_image_grid(X; nrows=4, ncols=4)` to display a 4×4 grid of examples.

In [ ]:
# Problem 1a — load the data  (5 pts)
# --------------------------------------------------------------------------
# YOUR CODE HERE: call load_mnist_digit, store the result in X

X = missing;

println("Data matrix size: ", size(X))   # expected: (784, 100)

In [ ]:
# Problem 1b — pixel statistics  (5 pts)
# --------------------------------------------------------------------------
# YOUR CODE HERE: compute mean and std of all pixel values in X

μ_data = missing;
σ_data = missing;

@printf("Mean pixel value: %.4f\n", μ_data);
@printf("Std  pixel value: %.4f\n", σ_data);

In [ ]:
# Problem 1c — visualise training examples  (5 pts)
show_image_grid(X; nrows=4, ncols=4)

___
## Problem 2 — Autoencoder Architecture  (30 points)

Implement the two core operations of the autoencoder forward pass.

| Function | Input → Output | Points |
|----------|---------------|--------|
| `encode(model, x)` | $(D\times N)$ data → bottleneck codes $\mathbf{z}$, $(L\times N)$ | 15 |
| `decode(model, z)` | $(L\times N)$ codes → reconstruction $\hat{\mathbf{x}}$, $(D\times N)$ | 15 |

The `MyAEModel` struct and `build_ae_model` constructor are provided — run the cell below
to create the model, then implement the two functions.

In [ ]:
# Build the autoencoder — nothing to change here
ae = build_ae_model(D, H, L);
println("AE created.")
println("  Encoder : ", ae.encoder)
println("  Decoder : ", ae.decoder)

In [ ]:
# Problem 2a — implement encode  (15 pts)
# --------------------------------------------------------------------------
# encode maps a batch of inputs x (D × N) to bottleneck codes z (L × N).
#
# Steps:
#   1. Pass x through model.encoder
#   2. Return the result

function encode(model::MyAEModel, x::AbstractMatrix)
    # YOUR CODE HERE
    
end

In [ ]:
# Problem 2b — implement decode  (15 pts)
# --------------------------------------------------------------------------
# decode maps bottleneck codes z (L × N) to reconstructed inputs x̂ (D × N).
#
# Steps:
#   1. Pass z through model.decoder
#   2. Return the result
#      (the final sigmoid layer already constrains output to [0, 1])

function decode(model::MyAEModel, z::AbstractMatrix)
    # YOUR CODE HERE
    
end

In [ ]:
# Unit tests — all @test lines should pass before moving to Problem 3
let
    x_test = X[:, 1:5];
    z_test = encode(ae, x_test);
    x̂_test = decode(ae, z_test);

    @test size(z_test) == (L, 5);          # encode: correct bottleneck shape
    @test size(x̂_test) == (D, 5);         # decode: correct reconstruction shape
    @test all(0f0 .≤ x̂_test .≤ 1f0);    # decode: sigmoid output in [0, 1]

    println("✓  All Problem 2 tests passed!");
end

___
## Problem 3 — Reconstruction Loss and Training  (30 points)

### 3a  (15 pts) — Implement `reconstruction_loss`

Implement the mean-squared reconstruction loss:

$$\mathcal{L}(\theta,\phi;\,\mathbf{X}) = \frac{1}{N}\sum_{i=1}^{N}\|\mathbf{x}_i - \hat{\mathbf{x}}_i\|^2$$

where $\hat{\mathbf{x}}_i = \text{decode}(\text{encode}(\mathbf{x}_i))$.

**Hint:** `mean(sum((x .- x̂).^2; dims=1))` — sum squared errors over the $D$ pixel
dimensions, then average over the $N$ examples in the batch.

### 3b  (15 pts) — Complete the training loop

Fill in the single missing line inside `Flux.withgradient` in the training loop below.

In [ ]:
# Problem 3a — implement reconstruction_loss  (15 pts)
# --------------------------------------------------------------------------
# reconstruction_loss should:
#   1. Encode x  →  z   using encode(model, x)
#   2. Decode z  →  x̂  using decode(model, z)
#   3. Return the MSE:  mean(sum((x .- x̂).^2; dims=1))

function reconstruction_loss(model::MyAEModel, x::AbstractMatrix)
    # YOUR CODE HERE
    
end

In [ ]:
# Sanity-check the loss
let
    x_test = X[:, 1:5];
    loss   = reconstruction_loss(ae, x_test);

    @test loss isa AbstractFloat;   # returns a scalar
    @test loss > 0;                 # MSE is non-negative
    @test loss < D;                 # sanity: less than the max possible MSE

    @printf("  Reconstruction loss (untrained): %.4f\n", loss);
    println("✓  reconstruction_loss tests passed!");
end

In [ ]:
# Problem 3b — complete the training loop  (15 pts)
# --------------------------------------------------------------------------
opt_state = Flux.setup(Adam(LR), ae);
losses    = Float32[];

println("Training autoencoder...");
for epoch in 1:NUM_EPOCHS
    loss, grads = Flux.withgradient(ae) do m
        # YOUR CODE HERE — one line: call reconstruction_loss on the full dataset X
        
    end;
    Flux.update!(opt_state, ae, grads[1]);
    push!(losses, loss);
    epoch % 400 == 0 && @printf("  Epoch %4d | loss = %.4f\n", epoch, loss);
end
println("Training complete.");

In [ ]:
plot(losses;
    xlabel="Epoch", ylabel="Reconstruction Loss (MSE)",
    title="Autoencoder Training", label="MSE",
    color=:steelblue, lw=2, framestyle=:box)

___
## Problem 4 — Latent Space Analysis  (25 points)

### 4a  (10 pts) — Reconstruction quality
Run the cell below to compare original images with their reconstructions.

### 4b  (15 pts) — Latent-space interpolation
A useful test of whether the autoencoder has learned a *smooth* latent space is to
linearly interpolate between the bottleneck codes of two training images and decode
the path.

Complete the interpolation cell below:
1. Encode `x1` and `x2` to get `z1` and `z2` (each `L × 1`)
2. For `n_steps = 10` evenly-spaced values of $\alpha\in[0,1]$, compute
   $\mathbf{z}_\alpha = (1-\alpha)\,\mathbf{z}_1 + \alpha\,\mathbf{z}_2$
3. Stack the interpolated codes into a `(L × n_steps)` matrix and decode in one call
4. Display the result with `show_image_grid(x_path; nrows=2, ncols=5)`

Then answer the discussion questions.

In [ ]:
# Problem 4a — reconstruction quality  (10 pts)
let
    n_show  = 8;
    x_orig  = X[:, 1:n_show];
    z_orig  = encode(ae, x_orig);
    x_recon = decode(ae, z_orig);

    p_orig  = show_image_grid(x_orig;  nrows=2, ncols=4);
    p_recon = show_image_grid(x_recon; nrows=2, ncols=4);
    plot(p_orig, p_recon; layout=(1, 2), size=(700, 250),
         plot_title="Left: originals   Right: reconstructions")
end

In [ ]:
# Problem 4b — latent-space interpolation  (15 pts)
# --------------------------------------------------------------------------
n_steps = 10;
x1 = X[:, 1:1];   # first training image   (784 × 1)
x2 = X[:, 6:6];   # sixth training image   (784 × 1)

# YOUR CODE HERE
# 1. z1 = encode(ae, x1),  z2 = encode(ae, x2)
# 2. For each α in range(0f0, 1f0, length=n_steps): z_α = (1-α)*z1 + α*z2
# 3. Stack into z_path  (L × n_steps),  then x_path = decode(ae, z_path)
# 4. Call show_image_grid(x_path; nrows=2, ncols=5)

z1 = missing;
z2 = missing;

alphas = range(0f0, 1f0; length=n_steps);
z_path = missing;   # L × n_steps matrix of interpolated codes
x_path = missing;   # decode all at once

show_image_grid(x_path; nrows=2, ncols=5)

### Discussion Questions  (included in Problem 4b score)

*Edit this cell to answer in 2–4 sentences each.*

---

**Q1.** Looking at your reconstruction panel (Problem 4a), are the reconstructions sharp
or blurry?  Given that the bottleneck dimension is only $L=8$ while the input is
$D=784$, what information must the encoder have discarded?

> *Your answer here.*

---

**Q2.** The interpolation in Problem 4b walks a straight line through the $L=8$
dimensional bottleneck space.  Does the image sequence transition smoothly from the first
digit to the sixth, or do intermediate frames look unrecognisable?  What does this tell
you about whether the autoencoder has learned a **smooth** latent space?

> *Your answer here.*

---

**Q3.** A standard autoencoder has no explicit constraint on the shape of the latent
space — any bottleneck code that minimises the reconstruction loss is acceptable.
If you tried to generate a *new* digit by sampling $\mathbf{z}$ randomly from, say,
$\mathcal{N}(\mathbf{0},\mathbf{I})$ and decoding it, would you expect the output to
look like a realistic digit?  Why or why not?

> *Your answer here.*